# Force closure
- Cartesian Impedance position control + PI force control

> Flow
1. Declare xml: box with contact site (L/R)
2. Get targets: contact position, contact force direction, contact force magnitude
3. Define each controller: position with PD, contact force with PI

#### 0. Generate Scene

In [1]:
import os
import sys
import numpy as np
import time
import mujoco
sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *
from pp_base_mujoco.SPEC_HELPER import *
from pp_base_mujoco.KINEMATICS import *

In [2]:
spec_helper = MjSpecHelper()
spec_helper.add_robot(
    path='../asset/panda/panda_ee_sphere.xml',
    body_name="base",
    p=(0, 0.7, 0),
    # r=(0, 0, -1.57),
    r=(0, 0, 0),
    prefix="",
    suffix="_right"
)
spec_helper.add_robot(
    path='../asset/panda/panda_ee_sphere.xml',
    body_name="base",
    p=(0, -0.7, 0),
    # r=(0, 0, 1.57),
    r=(0, 0, 0),
    prefix="",
    suffix="_left"
)
spec_helper.add_geom(
    name="box",
    type='box',
    size=(0.15, 0.15, 0.15),
    freejoint = False,
    p=(0.2, 0, 0.15),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 0.3, 0.5),
    group=1,
    friction=(1.0, 0.005, 0.0001),
    mass=0.5
)
spec_helper.add_site(
    name="contact_right",
    size=(0.05,0.05,0.05),
    p=(0, 0.15+0.05, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)
spec_helper.add_site(
    name="contact_left",
    size=(0.05,0.05,0.05),
    p=(0, -0.15-0.05, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)
spec_helper.add_site(
    name="center",
    size=(0.05,0.05,0.05),
    p=(0, 0, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)

model, data = spec_helper.compile()
spec_helper.save_to_xml("../asset/xml/scene_panda_lr.xml")

#### 1. Initialize Scene

In [3]:
joint_names = get_joint_names(model, data)
joint_names_left = [name for name in joint_names if name is not None and "_left" in name]
joint_names_right = [name for name in joint_names if name is not None and "_right" in name]

""" GO TO INITIAL QPOS """
qpos_init = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0.5]) # set initial qpos
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

In [4]:
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

while viewer.is_alive():
    # mujoco.mj_step(model, data)
    mujoco.mj_kinematics(model, data)
    # mujoco.mj_forward(model, data)
    viewer.render()

viewer.close()
del(viewer)

#### 2. Get EE target & IK

In [5]:
# current end effector position rotation 
p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
print("Current EE position (left):", p_ee_left)
print("Current EE position (right):", p_ee_right)
# rotation 
R_ee_left = get_R(model, data, name="eef_sphere_left", type='geom')
R_ee_right = get_R(model, data, name="eef_sphere_right", type='geom')
print("Current EE rotation (left):", R_ee_left)
print("Current EE rotation (right):", R_ee_right)

Current EE position (left): [ 0.39240442 -0.7         0.45858535]
Current EE position (right): [0.39240442 0.7        0.45858535]
Current EE rotation (left): [[ 9.59410820e-01  2.82012194e-01  1.02892521e-16]
 [ 2.82012194e-01 -9.59410820e-01 -1.78037586e-17]
 [ 9.36953212e-17  4.60980644e-17 -1.00000000e+00]]
Current EE rotation (right): [[ 9.59410820e-01  2.82012194e-01  1.02892521e-16]
 [ 2.82012194e-01 -9.59410820e-01 -1.78037586e-17]
 [ 9.36953212e-17  4.60980644e-17 -1.00000000e+00]]


In [6]:
# contact position 
p_target_contact_left = get_p(model, data, name="contact_left", type='site')
p_target_contact_right = get_p(model, data, name="contact_right", type='site')
print("Contact target position (left):", p_target_contact_left)
print("Contact target position (right):", p_target_contact_right)

R_target_contact_left = [[1.0, 0.0, 0.0],[0.0, 0.0, 1.0],[0.0, -1.0, 0.0]]
R_target_contact_right = [[1.0, 0.0, 0.0],[0.0, 0.0, -1.0],[0.0, 1.0, 0.0]]
print("Target contact frame rotation (left):", R_target_contact_left)
print("Target contact frame rotation (right):", R_target_contact_right)

Contact target position (left): [ 0.2  -0.2   0.15]
Contact target position (right): [0.2  0.2  0.15]
Target contact frame rotation (left): [[1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]]
Target contact frame rotation (right): [[1.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]]


In [7]:
# get error 
pos_error_left, rotvec_error_left = get_ik_error_clipped(
    p_target=p_target_contact_left,
    r_target=R_target_contact_left,
    p_current=p_ee_left,
    r_current=R_ee_left,)
print("Position error (left):", pos_error_left)
print("Rotation error (left):", rotvec_error_left)

pos_error_right, rotvec_error_right = get_ik_error_clipped(
    p_target=p_target_contact_right,
    r_target=R_target_contact_right,
    p_current=p_ee_right,
    r_current=R_ee_right,)
print("Position error (right):", pos_error_right)
print("Rotation error (right):", rotvec_error_right)

Position error (left): [-0.02  0.02 -0.02]
Rotation error (left): [ 0.2         0.12790796 -0.12790796]
Position error (right): [-0.02 -0.02 -0.02]
Rotation error (right): [-0.2        -0.12790796 -0.12790796]


In [8]:
def get_jacobian_franka_ee_bimanual():
    jacobian_p_right, jacobian_r_right = get_jacobian(model, data, 'eef_sphere_right', type='geom')
    # reduce jacobian of 3x14 to 3x7, first half for right arm 
    jacobian_p_right = jacobian_p_right[:, :7]
    jacobian_r_right = jacobian_r_right[:, :7]
    jacobian_p_left, jacobian_r_left = get_jacobian(model, data, 'eef_sphere_left', type='geom')
    # reduce jacobian of 3x14 to 3x7, second half for left arm
    jacobian_p_left = jacobian_p_left[:, 7:]
    jacobian_r_left = jacobian_r_left[:, 7:]

    return jacobian_p_left, jacobian_r_left, jacobian_p_right, jacobian_r_right

#### 3. Reach target EE pR with IK

In [9]:
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

while viewer.is_alive():
    # get current EE position & rotation
    p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
    p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
    R_ee_left = get_R(model, data, name="eef_sphere_left", type='geom')
    R_ee_right = get_R(model, data, name="eef_sphere_right", type='geom')
    # get target p (R is globally fixed )
    p_target_contact_left = get_p(model, data, name="contact_left", type='site')
    p_target_contact_right = get_p(model, data, name="contact_right", type='site')
    # calculate ik error 
    pos_error_left, rotvec_error_left = get_ik_error_clipped(
        p_target=p_target_contact_left,
        r_target=R_target_contact_left,
        p_current=p_ee_left,
        r_current=R_ee_left,
        )
    pos_error_right, rotvec_error_right = get_ik_error_clipped(
        p_target=p_target_contact_right,
        r_target=R_target_contact_right,
        p_current=p_ee_right,
        r_current=R_ee_right,
        )
    error_left = np.concatenate([pos_error_left, rotvec_error_left])
    error_right = np.concatenate([pos_error_right, rotvec_error_right])

    # terminate condition
    if np.linalg.norm(pos_error_left) < 0.01 and np.linalg.norm(rotvec_error_left) < 0.01:
        qpos_left = get_qpos_with_names(model, data, names=joint_names_left)
        # print("\r left arm Target reached!", end="")
    if np.linalg.norm(rotvec_error_right) < 0.01 and np.linalg.norm(pos_error_right) < 0.01:
        qpos_right = get_qpos_with_names(model, data, names=joint_names_right)
        print("\r right arm Target reached!", end="")

    # calculate jacobian
    jac_p_left, jac_r_left, jac_p_right, jac_r_right = get_jacobian_franka_ee_bimanual() 
    jac_left = np.concatenate([jac_p_left, jac_r_left], axis=0)
    jac_right = np.concatenate([jac_p_right, jac_r_right], axis=0)
    # jac_left_inverse = get_pseudo_inverse(jac_left, method="dls", damping=0.1)
    # jac_right_inverse = get_pseudo_inverse(jac_right, method="dls", damping=0.1)
    jac_left_inverse = get_pseudo_inverse(jac_left, method="svd", sigma_threshold=1e-3)
    jac_right_inverse = get_pseudo_inverse(jac_right, method="svd", sigma_threshold=1e-3)
    # calculate qpos error
    qpos_error_left = jac_left_inverse @ error_left
    qpos_error_right = jac_right_inverse @ error_right
    q_left_updated = get_qpos_with_names(model, data, names=joint_names_left) + qpos_error_left
    q_right_updated = get_qpos_with_names(model, data, names=joint_names_right) + qpos_error_right
    # update qpos
    apply_qpos_names(model, data, names=joint_names_left, value=q_left_updated)
    apply_qpos_names(model, data, names=joint_names_right, value=q_right_updated)
    mujoco.mj_forward(model, data)
    viewer.render()
    time.sleep(0.05) # for visualization stability

# close
viewer.close()
del(viewer)

 right arm Target reached!

#### 4-1. End effector position: Cartesian Impedance controller
- Declare gains & jacobians
- position targets & differences

In [10]:
# Gains
Kp_ee = 100.0
Kd_ee = 20.0
Kp_force = 10.0
Ki_force = 1.0
# get jacobian transpose
jac_p_left, jac_R_left, jac_p_right, jac_R_Right = get_jacobian_franka_ee_bimanual()
jac_left = np.concatenate([jac_p_left, jac_R_left], axis=0)
jac_right = np.concatenate([jac_p_right, jac_R_Right], axis=0)
# pseudo inverse of positional jacobian
jac_left_inverse = get_pseudo_inverse(jac_left, method="svd", sigma_threshold=1e-3)
jac_right_inverse = get_pseudo_inverse(jac_right, method="svd", sigma_threshold=1e-3)

In [11]:
print("jac p left shape:", jac_p_left.shape)
print("jac R left shape:", jac_R_left.shape)
print("jac p right shape:", jac_p_right.shape)
print("jac R right shape:", jac_R_Right.shape)

jac p left shape: (3, 7)
jac R left shape: (3, 7)
jac p right shape: (3, 7)
jac R right shape: (3, 7)


In [16]:
# get positional error
p_ee_target_left = get_p(model, data, name="contact_left", type='site')
p_ee_target_right = get_p(model, data, name="contact_right", type='site')
p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
p_ee_left_error = p_ee_target_left - p_ee_left
p_ee_right_error = p_ee_target_right - p_ee_right

v_ee_target_left = np.zeros(3)
v_ee_target_right = np.zeros(3)
qvel_left = get_qvel_with_names(model, data, names=joint_names_left)
qvel_right = get_qvel_with_names(model, data, names=joint_names_right)
v_ee_left = jac_p_left @ qvel_left
v_ee_right = jac_p_right @ qvel_right
v_ee_left_error = v_ee_target_left - v_ee_left
v_ee_right_error = v_ee_target_right - v_ee_right

# calculate torque 
f_ee_desired_left = Kp_ee * p_ee_left_error + Kd_ee * v_ee_left_error
f_ee_desired_right = Kp_ee * p_ee_right_error + Kd_ee * v_ee_right_error
torque_left = f_ee_desired_left @ jac_left_inverse.T
torque_right = f_ee_desired_right @ jac_right_inverse.T

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 6 is different from 3)

#### 4-2. Push force: PI controller for cartesian force
- Force direction target with site positions
- Force magnitude: calculate with friction

In [ ]:
def get_body_contact_force_position(
        model,
        data,
        body1_name,
        body2_name,
        ):
    body1_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body1_name)
    body2_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body2_name)
    contact_forces = []
    contact_positions = []
    for i in range(data.ncon):
        contact = data.contact[i]
        body1_in_contact_id = model.geom_bodyid[contact.geom1]
        body2_in_contact_id = model.geom_bodyid[contact.geom2]
        if (body1_in_contact_id == body1_id and body2_in_contact_id == body2_id) or (body1_in_contact_id == body2_id and body2_in_contact_id == body1_id):
            force = np.zeros(6) # (6,)
            mujoco.mj_contactForce(model,data,i,force)
            contact_forces.append(force)
            contact_positions.append(contact.pos)
    return contact_forces, contact_positions

In [ ]:
# get desired cartesian force direction & magnitude 
p_center = get_p(model, data, name="center", type='site')
direction_push_left = (p_center - p_target_contact_left) / np.linalg.norm(p_center - p_target_contact_left)
direction_push_right = (p_center - p_target_contact_right) / np.linalg.norm(p_center - p_target_contact_right)
force_magnitude = 5.0 # 5 newtons 
f_push_target_left = Kp_force * (force_magnitude * direction_push_left)
f_push_target_right = Kp_force * (force_magnitude * direction_push_right)
# get current contact force
f_push_left = get_body_contact_force_position(model, data, body1_name="eef_sphere_left", body2_name="box")
f_push_right = get_body_contact_force_position(model, data, body1_name="eef_sphere_right", body2_name="box")
print("Current contact force (left):", f_push_left)
print("Current contact force (right):", f_push_right)

# desired force with PI controller
f_accumulated_left = np.zeros(3)
f_accumulated_right = np.zeros(3)
f_push_error_left = f_push_target_left - f_push_left[0]
f_push_error_right = f_push_target_right - f_push_right[0]
f_accumulated_left += f_push_error_left * 0.01 # integral term with dt=0.01s
f_accumulated_right += f_push_error_right * 0.01

# get torque
f_push_desired_left = f_push_target_left + Kp_force * f_push_error_left + Ki_force * f_accumulated_left
f_push_desired_right = f_push_target_right + Kp_force * f_push_error_right + Ki_force * f_accumulated_right
torque_left = jac_left_inverse.T @ f_push_desired_left
torque_right = jac_right_inverse.T @ f_push_desired_right

#### 5. Iterate
- Iteratively apply two torques

In [ ]:
# Required force -> torque mapping with PI controller 
force_required_left = np.array([0, 0, 5])
force_required_right = np.array([0, 0, -5])
# apply PI control to map to torque 


# get jacobian, mapping to torque 
jac_p_left, _, jac_p_right, _ = get_jacobian_franka_ee_bimanual()
torque_left = jac_p_left.T @ force_required_left
torque_right = jac_p_right.T @ force_required_right